In [ ]:
# Example 1: low-level serialize/deserialize roundtrip via TensorSpec.
# Current API: serialize takes name -> TensorSpec(dtype, shape, data_ptr, data_len).
import numpy as np
from safetensors import TensorSpec, serialize, deserialize

arr = np.array([[1.0, 2.0], [3.0, 4.0]], dtype=np.float32)
spec = TensorSpec(dtype="float32", shape=list(arr.shape), data_ptr=arr.ctypes.data, data_len=arr.nbytes)

# `arr` must stay alive across serialize (raw-pointer contract).
blob = serialize({"weight": spec})
print(f"serialized: {len(blob)} bytes")

decoded = deserialize(blob)
print(f"decoded entries: {len(decoded)}")
name, info = decoded[0]
print(f"name={name} dtype={info['dtype']} shape={info['shape']} data_len={len(info['data'])}")
assert name == "weight"
assert info["shape"] == [2, 2]
assert info["dtype"] == "F32"
assert bytes(info["data"]) == arr.tobytes()

In [ ]:
# Example 2: numpy save/load roundtrip via safetensors.numpy.
import os
import tempfile

import numpy as np
from safetensors.numpy import save_file, load_file, save, load

tensors = {
    "a": np.arange(6, dtype=np.float32).reshape(2, 3),
    "b": np.array([[1, 2], [3, 4]], dtype=np.int64),
}

with tempfile.TemporaryDirectory() as d:
    path = os.path.join(d, "model.safetensors")
    save_file(tensors, path)
    loaded = load_file(path)
    print({k: (v.dtype, v.shape) for k, v in loaded.items()})
    assert np.array_equal(loaded["a"], tensors["a"])
    assert np.array_equal(loaded["b"], tensors["b"])

# In-memory variant: save -> bytes -> load.
blob = save(tensors)
loaded_mem = load(blob)
assert np.array_equal(loaded_mem["a"], tensors["a"])
print(f"in-memory blob bytes: {len(blob)}")


In [ ]:
# Example 3: header / metadata parsing on raw bytes via deserialize.
import numpy as np
from safetensors.numpy import save
from safetensors import deserialize

tensors = {
    "w1": np.arange(12, dtype=np.float32).reshape(3, 4),
    "w2": np.arange(8, dtype=np.int32).reshape(2, 4),
}
blob = save(tensors)

entries = deserialize(blob)
print(f"num entries: {len(entries)}")
for name, info in entries:
    print(f"  {name}: dtype={info['dtype']} shape={info['shape']} bytes={len(info['data'])}")

names = {n for n, _ in entries}
assert names == {"w1", "w2"}
by_name = dict(entries)
assert by_name["w1"]["shape"] == [3, 4]
assert by_name["w2"]["dtype"] == "I32"


In [ ]:
# Example 4: error handling — corrupted header + missing-key access.
import os
import tempfile

import numpy as np
from safetensors import SafetensorError, safe_open, deserialize
from safetensors.numpy import save_file

# 4a) Corrupted blob: garbage bytes should raise.
raised_corrupt = False
try:
    deserialize(b"this-is-not-a-safetensors-file")
except SafetensorError as e:
    raised_corrupt = True
    assert "deserializing" in str(e), f"unexpected message: {e!r}"
    print(f"corrupt-bytes rejected as expected ({type(e).__name__})")
assert raised_corrupt, "deserialize should reject garbage bytes"

# 4b) safe_open with missing key should raise on get_tensor.
with tempfile.TemporaryDirectory() as d:
    path = os.path.join(d, "m.safetensors")
    save_file({"x": np.ones((2,), dtype=np.float32)}, path)
    raised_missing = False
    with safe_open(path, framework="numpy") as f:
        try:
            f.get_tensor("does_not_exist")
        except SafetensorError as e:
            raised_missing = True
            assert "does_not_exist" in str(e), f"unexpected message: {e!r}"
            print(f"missing-key rejected as expected ({type(e).__name__})")
    assert raised_missing, "safe_open.get_tensor should reject unknown keys"

print("error-handling checks passed")
